[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/kadimetla/AIML_Learning_Gang/blob/main/statistics/regression_methods/05_random_forest_regressor/Random_Forest_Regressor.ipynb)](https://colab.research.google.com/github/kadimetla/AIML_Learning_Gang/blob/main/statistics/regression_methods/05_random_forest_regressor/Random_Forest_Regressor.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/kadimetla/AIML_Learning_Gang/blob/main/statistics/regression_methods/05_random_forest_regressor/Random_Forest_Regressor.ipynb)

# Notebook 05 — Random Forest Regressor
## Averaging Many Trees for Stable Predictions

**Dataset**: California Housing
**Question**: *What is the median house value — using an ensemble of regression trees?*
**Type**: Supervised Regression — Ensemble

---

## Section 1 — What Is a Random Forest Regressor?

### In plain English

One surveyor estimating house values can be unreliable.
What if you sent **100 different surveyors**, each trained on a slightly different
set of districts, and then **averaged their estimates**?

That is Random Forest Regression — 100 decision tree regressors, each trained on a
random bootstrap sample of the training data, with a random subset of features at each split.

For a new district:
- Each tree produces a predicted house value
- The forest returns the **average** of all predictions

### Why averaging beats a single tree

| | Single Tree | Random Forest |
|---|---|---|
| Overfit risk | High | Low (averaging cancels noise) |
| R² on Titanic/Housing | ~0.60 | ~0.80 |
| Interpretability | High (print the tree) | Medium (feature importances) |
| Sensitivity to outliers | High | Low |

## Section 2 — How Does It Learn?

**For each tree (100 by default):**
1. Draw a bootstrap sample of training districts (random, with replacement)
2. Build a full regression tree on that sample
3. At each split, only consider `√(n_features)` random features

**Prediction:**
```
forest_prediction = (1 / N_trees) × Σ tree_i_prediction
```
Each tree memorises slightly different noise. The average keeps the signal and cancels the noise.

## Section 3 — What Does the Data Need?

### Tree-based regressors do not need scaling

Decision trees split on thresholds — the absolute scale of features is irrelevant.

| Feature | Transform for trees |
|---|---|
| `MedInc` | None — raw values work |
| `HouseAge` | None |
| `AveRooms` | Clip extreme outliers (optional) |
| `AveBedrms` | Clip extreme outliers (optional) |
| `Population` | None |
| `AveOccup` | Clip extreme outliers |
| `Latitude` | None |
| `Longitude` | None |

> Applying StandardScaler to a tree model produces **identical accuracy** — trees are invariant to
> monotonic transformations. Skipping it makes the tree thresholds easier to interpret.

In [ ]:
from sklearn.datasets import fetch_california_housing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseVal'] = housing.target
for col in ['AveRooms', 'AveBedrms', 'AveOccup']:
    df[col] = df[col].clip(upper=df[col].quantile(0.99))

FEATURES = housing.feature_names.tolist()
TARGET   = 'MedHouseVal'

print(f"California Housing: {df.shape[0]} districts, {len(FEATURES)} features")
print(f"Target (MedHouseVal): median house value in $100k, range {df[TARGET].min():.2f}–{df[TARGET].max():.2f}")
df.head(6)

In [ ]:
from sklearn.model_selection import train_test_split

X = df[FEATURES].values
y = df[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"Train: {len(X_train)} districts  |  Test: {len(X_test)} districts")
print("No scaling applied — tree-based models do not need it.")

## Section 4 — Train the Model and Read the Rules

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

single = DecisionTreeRegressor(max_depth=None, random_state=42)
single.fit(X_train, y_train)
rmse_tree = np.sqrt(mean_squared_error(y_test, single.predict(X_test)))
r2_tree   = r2_score(y_test, single.predict(X_test))

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred  = rf.predict(X_test)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred))
r2_rf   = r2_score(y_test, y_pred)

print(f"Single Decision Tree (unlimited): RMSE={rmse_tree:.4f}  R²={r2_tree:.4f}")
print(f"Random Forest (100 trees)       : RMSE={rmse_rf:.4f}  R²={r2_rf:.4f}")
print(f"Improvement in R²: +{(r2_rf - r2_tree):.4f}")

In [ ]:
importance_df = pd.DataFrame({
    'Feature': FEATURES, 'Importance': rf.feature_importances_.round(4)
}).sort_values('Importance', ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.barh(importance_df['Feature'], importance_df['Importance'], color='#26A69A')
ax1.set_title('Random Forest Feature Importances\n(averaged across 100 trees)', fontsize=12)
ax1.set_xlabel('Mean MSE reduction', fontsize=11)

ax2.scatter(y_test, y_pred, alpha=0.15, s=10, color='steelblue')
lim = max(y_test.max(), y_pred.max()) * 1.02
ax2.plot([0, lim], [0, lim], 'r--', lw=1.5, label='Perfect prediction')
ax2.set_xlabel('Actual value (×$100k)', fontsize=11)
ax2.set_ylabel('Predicted value (×$100k)', fontsize=11)
ax2.set_title(f'Actual vs Predicted\nRMSE={rmse_rf:.3f}  R²={r2_rf:.3f}', fontsize=12)
ax2.legend()

plt.tight_layout()
plt.show()